# 🧠 Personal Knowledge Worker V2 - Advanced RAG

This improved version adds enterprise-grade RAG features:
- **t-SNE Visualization**: Interactive Plotly dashboard to see your data clusters.
- **Intelligent Indexing**: MD5-based versioning to avoid redundant embedding costs.
- **Direct File Ingestion**: Ingests whole directories of Markdown/PDF files.
- **Local/Cloud Hybrid**: Switch between OpenAI and Local Embeddings.

In [ ]:
import os
import hashlib
import glob
import numpy as np
import plotly.graph_objects as go
from sklearn.manifold import TSNE
from openai import OpenAI
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import CharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from dotenv import load_dotenv

load_dotenv(override=True)
client = OpenAI()
DB_NAME = "v2_vector_db"
DATA_DIR = "knowledge-base" # Directory with your .md files

## 1. Intelligent Indexing with Content Hashing

We only rebuild the database if the underlying files have changed.

In [ ]:
def get_dir_hash(directory):
    """Generates a combined MD5 hash of all markdown files in the directory."""
    hasher = hashlib.md5()
    files = sorted(glob.glob(os.path.join(directory, "**/*.md"), recursive=True))
    for f in files:
        with open(f, "rb") as file_obj:
            hasher.update(file_obj.read())
    return hasher.hexdigest()

def initialize_vector_store():
    current_hash = get_dir_hash(DATA_DIR)
    hash_file = os.path.join(DB_NAME, "dir_hash.txt")
    
    embeddings = OpenAIEmbeddings()
    
    if os.path.exists(hash_file):
        with open(hash_file, "r") as f:
            saved_hash = f.read().strip()
        if saved_hash == current_hash:
            print("✅ Index is up to date. Loading from disk.")
            return Chroma(persist_directory=DB_NAME, embedding_function=embeddings)
    
    print("🔄 Changes detected or first run. Re-indexing documents...")
    loader = DirectoryLoader(DATA_DIR, glob="**/*.md", loader_cls=TextLoader)
    docs = loader.load()
    
    text_splitter = CharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    chunks = text_splitter.split_documents(docs)
    
    if os.path.exists(DB_NAME):
        import shutil
        shutil.rmtree(DB_NAME)
        
    vectorstore = Chroma.from_documents(
        documents=chunks, 
        embedding=embeddings, 
        persist_directory=DB_NAME
    )
    
    os.makedirs(DB_NAME, exist_ok=True)
    with open(hash_file, "w") as f:
        f.write(current_hash)
        
    return vectorstore

## 2. t-SNE Vector Visualization

Visualizing the semantic relationships between chunks.

In [ ]:
def visualize_vectors(vectorstore):
    collection = vectorstore._collection
    data = collection.get(include=['embeddings', 'documents', 'metadatas'])
    
    vectors = np.array(data['embeddings'])
    texts = data['documents']
    
    # Reduce dimension using t-SNE
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(vectors)-1))
    reduced = tsne.fit_transform(vectors)
    
    fig = go.Figure(data=[go.Scatter(
        x=reduced[:, 0],
        y=reduced[:, 1],
        mode='markers',
        text=[t[:100] for t in texts],
        marker=dict(size=8, color=np.arange(len(vectors)), colorscale='Viridis')
    )])
    
    fig.update_layout(title="Semantic Space Visualization (t-SNE)", template="plotly_dark")
    fig.show()

In [ ]:
if __name__ == "__main__":
    # Ensure DATA_DIR exists for demo purposes
    os.makedirs(DATA_DIR, exist_ok=True)
    
    db = initialize_vector_store()
    visualize_vectors(db)
    
    # Simple Query Logic
    # query = "Project goals?"
    # results = db.similarity_search(query, k=3)
    # print(results[0].page_content)